# Checking the five assumptions of linear regression

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tools import add_constant

In [2]:
df = pd.read_csv("data/data.csv")      # 200 rows, 3 inputs, 1 target
X, y = df.iloc[:, :3].to_numpy(), df["target"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
residual = y_test - y_pred            # error on each test row
print(model.coef_.round(2), round(model.score(X_test, y_test), 3))

[72.75 -0.28 53.34] 0.957


## 1. Linearity: each input against the target

In [3]:
for c in ["feature1", "feature2", "feature3"]:
    px.scatter(df, x=c, y="target", title=c).show()
df.corr()["target"].round(3)

feature1    0.819
feature2    0.031
feature3    0.578
target      1.000
Name: target, dtype: float64

## 2. No multicollinearity: VIF and correlations between inputs

In [4]:
Xc = add_constant(X_train)                      # VIF needs the intercept column
vif = [variance_inflation_factor(Xc, i) for i in range(1, Xc.shape[1])]
pd.Series(vif, index=df.columns[:3]).round(3)

feature1    1.011
feature2    1.010
feature3    1.014
dtype: float64

In [5]:
px.imshow(df.iloc[:, :3].corr().round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r")

## 3. Normality of residuals

In [6]:
print("skewness", round(stats.skew(residual), 3))
print("Shapiro-Wilk p-value", round(stats.shapiro(residual).pvalue, 3))   # above 0.05: no evidence against normality
px.histogram(x=residual, nbins=15, title="residuals")

skewness -0.228
Shapiro-Wilk p-value 0.506


In [7]:
(osm, osr), (slope, intercept, _) = stats.probplot(residual)
fig = px.scatter(x=osm, y=osr, labels={"x": "normal quantile", "y": "sorted residual"}, title="Q-Q plot")
fig.add_scatter(x=osm, y=slope * osm + intercept, mode="lines", name="normal line")
fig

## 4. Homoscedasticity: residual against prediction

In [8]:
px.scatter(x=y_pred, y=residual, labels={"x": "predicted", "y": "residual"})

## 5. No autocorrelation: residuals in order

In [9]:
print("Durbin-Watson", round(durbin_watson(residual), 2))   # near 2: no autocorrelation
px.line(y=residual, markers=True, labels={"index": "row", "y": "residual"})

Durbin-Watson 2.31


In [10]:
# Test: does a missing slow-changing input make residuals autocorrelated?
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(0); t = np.arange(200)
x = rng.normal(size=200); slow = np.sin(t / 25)
y_t = 2 * x + 3 * slow + rng.normal(size=200)
for name, Xt in [("without slow input", x[:, None]), ("with slow input", np.c_[x, slow])]:
    r = y_t - LinearRegression().fit(Xt, y_t).predict(Xt)
    print(name, "Durbin-Watson", round(durbin_watson(r), 2))

without slow input Durbin-Watson 0.4
with slow input Durbin-Watson 2.09
